In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
import numpy.random as rand
from tqdm import trange
import os,sys

from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement


sys.path.append('/home/austin/Basic')
from utils_tf import *
from utils_np import *
limitGPU(1024)



In [ ]:
sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data


## Make sure sampler works

In [ ]:
@tf.function
def dirichlet(g):
    N = len(g)
    gams = tf.random.gamma(shape=np.ones(N),alpha=g,beta=np.ones(N))
    sum_gams = tf.reduce_sum(gams)
    x = gams/sum_gams
    return x

In [ ]:
@tf.function
def kl_dirichlet(alpha,beta):
    alpha_0 = tf.reduce_sum(alpha)
    beta_0 = tf.reduce_sum(beta)
    term1 = tf.math.lgamma(alpha_0)
    term3 = tf.math.lgamma(beta_0)
    term2 = tf.reduce_sum(tf.math.lgamma(alpha))
    term4 = tf.reduce_sum(tf.math.lgamma(beta))
    diff = alpha - beta
    ddiff = tf.math.digamma(alpha)-tf.math.digamma(beta)
    term5 = tf.reduce_sum(diff*ddiff)
    return term1 - term2 - term3 + term4 + term5

In [ ]:
N = 10000
p = 3
samps = np.zeros((N,p))
g = np.ones(p).astype(np.float32)
for i in trange(N):
    samps[i] = dirichlet(g).numpy()

In [ ]:
plt.scatter(samps[:,0],samps[:,1],s=.1)
print(np.mean(samps,axis=0))

In [ ]:
N = 10000
p = 3
samps = np.zeros((N,p))
g = .1*np.ones(p).astype(np.float32)
for i in trange(N):
    samps[i] = dirichlet(g).numpy()
plt.scatter(samps[:,0],samps[:,1],s=.1)
print(np.mean(samps,axis=0))

In [ ]:
N = 10000
p = 3
samps = np.zeros((N,p))
g = 10*np.ones(p).astype(np.float32)
for i in trange(N):
    samps[i] = dirichlet(g).numpy()
plt.scatter(samps[:,0],samps[:,1],s=.1)
print(np.mean(samps,axis=0))

## Now read in the data

In [ ]:
fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_12.mat'
power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])
N = len(mouse)

indx_pos = (behaviornon1==1)&(condition==4)
indx_neg1 = (behaviornon1==2)&(condition==4)
indx_neg2 = (behaviornon1==2)&(condition==6)
indx_neg3 = (behaviornon1==2)&(condition==8)
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos

y = np.zeros(N)
y[indx_pos] = 1

mouse = mouse[indx_tot]
group = group[indx_tot]
expDate = expDate[indx_tot]
behavior = behavior[indx_tot]
behaviornon1 = behaviornon1[indx_tot]
time = time[indx_tot]
condition = condition[indx_tot]
y = y[indx_tot] 

N = len(mouse)


In [ ]:
training_set_idx = np.ones(N)
training_set_idx[mouse=='Mouse048'] = 0
training_set_idx[mouse=='Mouse7980'] = 0
training_set_idx[mouse=='Mouse7998'] = 0 

granger = np.exp(granger)
granger[granger>10] = 10
power = power*10
power[power>6] = 6

X = np.hstack((power,coherence,granger))
X = X[indx_tot]


In [ ]:
X_train = X[training_set_idx==1]
m_train = mouse[training_set_idx==1]
y_train = y[training_set_idx==1]

X_test = X[training_set_idx==0]
m_test = mouse[training_set_idx==0]
y_test = y[training_set_idx==0]
mice_training = np.unique(m_train)

In [ ]:
Xs_train = []
Ys_train = []
for i in range(len(mice_training)):
    myIdx = m_train==mice_training[i]
    Xs_train.append(X_train[myIdx==1])
    Ys_train.append(y_train[myIdx==1])

In [ ]:
for i in range(len(Xs_train)):
    print(Xs_train[i].shape,len(Ys_train[i]))

## Now read in the homecage data

In [ ]:
fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_time.mat'
powert,coherencet,grangert,labelst = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

In [ ]:
myLabelt = labelst['windows']
mouset = np.asarray(myLabelt['mouse'])
groupt = np.asarray(myLabelt['group'])
expDatet = np.asarray(myLabelt['expDate'])
N = len(mouset)
training_set_idxt = np.ones(N)
training_set_idxt[mouset=='Mouse048'] = 0
training_set_idxt[mouset=='Mouse7980'] = 0
training_set_idxt[mouset=='Mouse7998'] = 0 

grangert = np.exp(grangert)
grangert[grangert>10] = 10
powert = powert*10
powert[powert>6] = 6

Xt = np.hstack((powert,coherencet,grangert))


In [ ]:
def basic_batch_XYZ(bs,X,Y,Z):
    N = X.shape[0]
    bs2 = np.minimum(N,bs)
    idx = rand.choice(N,size=bs2,replace=False)
    X_batch = X[idx]
    Y_batch = Y[idx]
    Z_batch = Z[idx]
    return X_batch.astype(np.float32),Y_batch.astype(np.float32),Z_batch.astype(np.float32)

def basic_batch_X2(bs,X):
    N = X.shape[0]
    bs2 = np.minimum(N,bs)
    idx = rand.choice(N,size=bs2,replace=False)
    X_batch = X[idx]
    return X_batch.astype(np.float32)

def basic_batch_XY2(bs,X,Y):
    N = X.shape[0]
    bs2 = np.minimum(N,bs)
    idx = rand.choice(N,size=bs2,replace=False)
    X_batch = X[idx]
    Y_batch = Y[idx]
    return X_batch.astype(np.float32),Y_batch.astype(np.float32)

In [ ]:
Xt_train = Xt[training_set_idxt==1]
Xt_test = Xt[training_set_idxt==0]
mt_train = mouset[training_set_idxt==1]
mt_test = mouset[training_set_idxt==0]

mice_t = np.unique(mt_train)

Xts_train = []
for i in range(len(mice_t)):
    myIdx = mt_train==mice_t[i]
    Xts_train.append(Xt_train[myIdx])


In [ ]:
for i in range(len(Xts_train)):
    print(Xts_train[i].shape)

In [ ]:
nnode1 = 40
def getNetwork():
    g = keras.Sequential()
    g.add(keras.layers.InputLayer(Xt_train.shape[1]))
    g.add(keras.layers.Dropout(0.4))
    g.add(keras.layers.Dense(nnode1,activation='elu'))
    g.add(keras.layers.Dropout(0.4))
    g.add(keras.layers.Dense(1))  
    return g

In [ ]:
## Network for predicting memberships
p1 = getNetwork()
p2 = getNetwork()
p3 = getNetwork()
p4 = getNetwork()

#This estimates the posterior mean for each observation
q = keras.Sequential()
q.add(keras.layers.InputLayer(Xt_train.shape[1]))
q.add(keras.layers.Dropout(0.4))
q.add(keras.layers.Dense(100))
q.add(keras.layers.Dropout(0.4))
q.add(keras.layers.Dense(4))
q.add(keras.layers.Softmax())

In [ ]:
p1.summary()

In [ ]:
optimizer = keras.optimizers.Nadam(.001)
trainable_variables = (p1.trainable_variables + p2.trainable_variables + 
                       p3.trainable_variables + p4.trainable_variables + 
                       q.trainable_variables)

In [ ]:
nMice = len(Xts_train)

In [ ]:
def Loss_ce(y,y_hat):
    ce = tf.nn.sigmoid_cross_entropy_with_logits(labels=tf.squeeze(y),logits=tf.squeeze(y_hat))
    return tf.reduce_mean(ce)

In [ ]:
alpha = 4.0
N_info = 30
niter=1000
lloss_sup = np.zeros(niter)
lloss_kl = np.zeros(niter)
lloss = np.zeros(niter)
baseline = tf.constant(alpha*np.ones(4).astype(np.float32))
kl_weight = 0.1
for t in trange(niter):
    #Turn the homecage into list of data
    X_home_batch = []
    X_cond_batch = []
    Y_cond_batch = []
    for i in range(len(mice_t)):
        X_home_batch.append(basic_batch_X2(N_info,Xts_train[i]))
    for i in range(len(mice_t)):
        X_temp,Y_temp = basic_batch_XY2(30,Xs_train[i],Ys_train[i])
        X_cond_batch.append(X_temp)
        Y_cond_batch.append(Y_temp)
    #Now we need to actually get the mappings
    with tf.GradientTape() as tape:        
        gs = [q(X_home_batch[i]) for i in range(nMice)]
        gmeans = [tf.reduce_mean(gs[i],axis=0) for i in range(nMice)]# This is going to have the means for each
        g_draw = [tf.squeeze(dirichlet(N_info*gmeans[i])) for i in range(nMice)] #Draw from the weights
        
        kl_term = [kl_dirichlet(gmeans[i],baseline) for i in range(nMice)]
        #Now we are going to create predictions based on each population
        p1s = [tf.squeeze(p1(X_cond_batch[i])) for i in range(nMice)] # 30 x 1
        p2s = [tf.squeeze(p2(X_cond_batch[i])) for i in range(nMice)]
        p3s = [tf.squeeze(p3(X_cond_batch[i])) for i in range(nMice)]
        p4s = [tf.squeeze(p4(X_cond_batch[i])) for i in range(nMice)]
        
        #Now we need to stack the observations into a bunch of 3-dimensional tensors
        stacks = [tf.stack([p1s[i],p2s[i],p3s[i],p4s[i]]) for i in range(nMice)] # 4 x 30
        #muls = [tf.multiply(g_draw[i],stacks[i]) for i in range(nMice)] # 4 x 30 
        muls = [g_draw[i]*tf.transpose(stacks[i]) for i in range(nMice)] # 4 x 30 
        y_hats = [tf.reduce_mean(muls[i],axis=1) for i in range(nMice)]
        
        #Now compute the lossese
        losses = [Loss_ce(Y_cond_batch[i],y_hats[i]) for i in range(nMice)]
        loss_sup = tf.math.add_n(losses)
        kl_ess = tf.stack(kl_term)
        loss_kl = tf.reduce_mean(kl_ess)
        loss = loss_sup + kl_weight*loss_kl
    grad = tape.gradient(loss,trainable_variables)
    optimizer.apply_gradients(zip(grad,trainable_variables))
    lloss[t] = loss.numpy()
    lloss_sup[t] = loss.numpy()
    lloss_kl[t] = loss_kl.numpy()
        
        
        
        

In [ ]:
X_cond_batch[i].shape

In [ ]:
loss_vals.shape

In [ ]:
plt.plot(loss_vals)

In [ ]:
len(g_draw)

In [ ]:
gd = tf.stack(g_draw)
gdn = gd.numpy()

In [ ]:
gdn

In [ ]:
np.mean(gdn,axis=0)

In [ ]:
np.std(gdn,axis=0)

In [ ]:
gm = tf.stack(gmeans)
print(gm.numpy())